# Fit a Jacobian lens on a free Colab GPU

Runtime > Change runtime type > **T4 GPU**, then run all cells.
Fits with the exact recipe from the repo configs (100 wikitext prompts,
seed 0, dim_batch 16, max_seq_len 128) and uploads the lens to the HF Hub.
Set `SLUG` below to the rung you want.

In [ ]:
SLUG = "qwen3-1.7b-base"
HF_LENS_REPO = "blzphnx/jlens-scaling-lenses"

!git clone -q https://github.com/blazingphoenix7/jlens-scaling
%cd jlens-scaling
!pip install -q . 2>&1 | tail -1
import sys
sys.path.insert(0, "/content/jlens-scaling/src")  # visible to the running kernel without restart
import torch
assert torch.cuda.is_available(), "switch the runtime to a T4 GPU first"
print(torch.cuda.get_device_name(0))

In [ ]:
from getpass import getpass
import huggingface_hub
huggingface_hub.login(token=getpass("HF write token: "))

In [ ]:
import torch, transformers, jlens
from jlens_scaling.config import load_config
from jlens_scaling.corpus import fitting_prompts

cfg = load_config(f"configs/{SLUG}.yaml")
hf = transformers.AutoModelForCausalLM.from_pretrained(
    cfg.model.model_id, revision=cfg.model.revision, dtype=torch.float32
).cuda()
tok = transformers.AutoTokenizer.from_pretrained(cfg.model.model_id)
model = jlens.from_hf(hf, tok)
prompts = fitting_prompts(cfg.fit, tok)
print(len(prompts), "prompts; n_layers", model.n_layers, "d_model", model.d_model)

In [ ]:
import logging, os
logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)
os.makedirs(f"artifacts/{SLUG}", exist_ok=True)
lens = jlens.fit(
    model, prompts,
    dim_batch=cfg.fit.dim_batch, max_seq_len=cfg.fit.max_seq_len,
    checkpoint_path=f"artifacts/{SLUG}/ckpt.pt", checkpoint_every=5,
)
lens.save(f"artifacts/{SLUG}/lens.pt")
print(lens)

In [ ]:
import json, time, jlens_scaling
meta = {
    "model_id": cfg.model.model_id,
    "revision_requested": cfg.model.revision,
    "revision_resolved": getattr(hf.config, "_commit_hash", None),
    "slug": cfg.model.slug,
    "n_prompts_requested": cfg.fit.n_prompts,
    "n_prompts_fitted": lens.n_prompts,
    "dim_batch": cfg.fit.dim_batch,
    "max_seq_len": cfg.fit.max_seq_len,
    "skip_first": jlens.fitting.SKIP_FIRST_N_POSITIONS,
    "corpus": cfg.fit.corpus,
    "corpus_seed": cfg.fit.corpus_seed,
    "min_tokens": cfg.fit.min_tokens,
    "source_layers": lens.source_layers,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "jlens_scaling_version": jlens_scaling.__version__,
    "fit_device": "colab-" + torch.cuda.get_device_name(0),
}
with open(f"artifacts/{SLUG}/fit_meta.json", "w") as f:
    json.dump(meta, f, indent=2)
!python scripts/upload_lenses.py --repo {HF_LENS_REPO}
print("done; lens is on the Hub")